# S.A.F.E. — Sistema de Asistencia para la Formación y Emergencias
### Agente de consulta sobre seguridad laboral para Triple T Corp

**Segunda evaluación · Ingeniería de Soluciones con Inteligencia Artificial**  
**Integrantes:** David Araya · Thomas Astete · Manuel Pinto

Este notebook presenta una demostración reproducible con datos simulados. Su configuración inicial conserva la secuencia del cuaderno base y continúa con herramientas, planificación, memoria, recuperación y pruebas. El prototipo informa y deriva; no autoriza trabajos ni reemplaza procedimientos vigentes.


# 0. Configuración del entorno

Este cuaderno sigue la secuencia inicial del notebook base: presentación, dependencias, importaciones, entorno, directorios, credenciales, dispositivo, parámetros y carga de modelos. Los datos de demostración se incorporan al cuaderno para evitar depender de rutas locales. Se conserva el modelo de embeddings multilingüe del proyecto; el servicio de chat se activa solo cuando hay una clave configurada.

El corpus contiene procedimientos simulados. No se debe usar para dirigir una tarea real.

### Instalación de dependencias

En Colab se instalan únicamente los paquetes ausentes, con rangos de versión para las bibliotecas principales. La primera instalación puede tardar. Si el runtime informa un conflicto de dependencias, reinícialo y ejecuta las celdas desde el comienzo.

In [ ]:
import importlib.util, subprocess, sys
EN_COLAB = "google.colab" in sys.modules
PAQUETES = {
    "langchain": "langchain>=1.0,<2.0",
    "langchain_openai": "langchain-openai>=1.0,<2.0",
    "langgraph": "langgraph>=1.0,<2.0",
    "langchain_community": "langchain-community>=0.4,<1.0",
    "sentence_transformers": "sentence-transformers>=3.0,<6.0",
    "faiss": "faiss-cpu>=1.8,<2.0",
}
FALTANTES = [paquete for modulo, paquete in PAQUETES.items()
             if importlib.util.find_spec(modulo) is None]
if EN_COLAB and FALTANTES:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", *FALTANTES])
    print("Se instalaron dependencias faltantes.")
elif EN_COLAB:
    print("Las dependencias principales ya están disponibles.")
else:
    print("Entorno local: no se instalaron paquetes automáticamente.")

Las dependencias principales ya están disponibles.


### Importaciones y preparación del entorno

Se usan bibliotecas del cuaderno base y componentes adicionales para el agente. Los datos y registros temporales se guardan en una carpeta de sesión creada al ejecutar; no se fija una ruta de Windows o de una cuenta personal.

In [ ]:
import os, json, tempfile
from pathlib import Path
import numpy as np

DIRECTORIO_SESION = Path(tempfile.mkdtemp(prefix="safe_ev2_"))
BASE = DIRECTORIO_SESION
DIRECTORIO_CACHE = DIRECTORIO_SESION / "cache"
DIRECTORIO_CACHE.mkdir(parents=True, exist_ok=True)
os.environ.setdefault("HF_HOME", str(DIRECTORIO_CACHE / "huggingface"))
TRACE_PATH = DIRECTORIO_SESION / "trace.jsonl"
MEMORY_PATH = DIRECTORIO_SESION / "long_term_memory.json"
print("Runtime Colab:", EN_COLAB)
print("Directorio temporal de esta sesión preparado.")

Runtime Colab: True
Directorio temporal de esta sesión preparado.


### Gestión de credenciales

En Colab, agrega `GROQ_API_KEY` en el panel **Secrets** (icono de llave) y habilita su acceso para este cuaderno. Se busca ese Secret primero, luego variables de entorno y, si está en la carpeta de trabajo o su carpeta inmediata, `API_KEY.txt`. El valor nunca se imprime ni se guarda en una celda.

Sin clave, se puede ejecutar el flujo local. Para ingresarla de forma oculta durante una sesión, ejecuta `solicitar_clave()` manualmente. En Colab también puedes subir `API_KEY.txt` con el panel de archivos.

In [ ]:
import getpass
API_KEY = None
if EN_COLAB:
    try:
        from google.colab import userdata
        API_KEY = userdata.get("GROQ_API_KEY") or userdata.get("LLM_API_KEY")
    except Exception:
        API_KEY = None
API_KEY = API_KEY or os.getenv("GROQ_API_KEY") or os.getenv("LLM_API_KEY")
if not API_KEY:
    for candidato in (Path.cwd()/"API_KEY.txt", Path.cwd().parent/"API_KEY.txt"):
        try:
            lineas = [x.strip() for x in candidato.read_text(encoding="utf-8").splitlines() if x.strip() and not x.lstrip().startswith("#")]
            if lineas:
                linea = lineas[0]
                API_KEY = linea.split("=",1)[1].strip().strip("\"'") if "=" in linea else linea.strip("\"'")
                break
        except (OSError, UnicodeError):
            pass
if API_KEY:
    os.environ["GROQ_API_KEY"] = API_KEY
    print("Credencial disponible; valor oculto.")
else:
    print("No se detectó clave. Continúa disponible la ruta local sin API.")

def solicitar_clave():
    """Solicita una clave oculta y la conserva solo en la sesión actual."""
    global API_KEY
    valor = getpass.getpass("Clave del proveedor (entrada oculta): ").strip()
    if valor:
        API_KEY = valor
        os.environ["GROQ_API_KEY"] = valor
        print("Clave configurada en memoria.")
    else:
        print("No se configuró una clave.")

No se detectó clave. Continúa disponible la ruta local sin API.


### Selección del dispositivo

Se usa GPU si PyTorch la detecta; de lo contrario, CPU. El modelo de chat no se descarga al runtime: si se configura, se consulta mediante el proveedor remoto. Los recursos disponibles pueden variar entre sesiones.

In [ ]:
try:
    import torch
    DISPOSITIVO = "cuda" if torch.cuda.is_available() else "cpu"
    if DISPOSITIVO == "cuda":
        try:
            libres, total = torch.cuda.mem_get_info()
            print(f"GPU disponible; memoria libre aproximada {libres/1024**3:.1f} de {total/1024**3:.1f} GiB")
        except Exception:
            print("GPU disponible.")
    else:
        print("No se detectó GPU CUDA; se usará CPU.")
except Exception:
    torch = None
    DISPOSITIVO = "cpu"
    print("PyTorch no está disponible; se seleccionó CPU para la ruta de respaldo.")
print("Dispositivo:", DISPOSITIVO)

GPU disponible; memoria libre aproximada 13.2 de 14.6 GiB
Dispositivo: cuda


### Parámetros y carga de modelos

Se conserva `paraphrase-multilingual-mpnet-base-v2`, utilizado por el notebook base. El modelo se carga una vez en la sesión y se usa en el dispositivo detectado. El chat mantiene el modelo Groq configurado en el proyecto; crear su cliente no ejecuta una llamada. Si la descarga de embeddings falla, se informa y se conserva la búsqueda léxica local.

In [ ]:
MODELO_EMBEDDINGS = "sentence-transformers/paraphrase-multilingual-mpnet-base-v2"
LLM_BASE_URL = os.getenv("LLM_BASE_URL", "https://api.groq.com/openai/v1")
LLM_MODEL = os.getenv("LLM_MODEL", "openai/gpt-oss-120b")
TEMPERATURA = 0.0
K_RETRIEVAL = 4
UMBRAL_SCORE = 0.07
MAX_TURNOS_MEMORIA = 4
try:
    from sentence_transformers import SentenceTransformer
    modelo_embeddings = SentenceTransformer(MODELO_EMBEDDINGS, device=DISPOSITIVO,
                                             cache_folder=str(DIRECTORIO_CACHE))
    print("Modelo de embeddings cargado en", DISPOSITIVO)
except Exception as error:
    modelo_embeddings = None
    print("Embeddings semánticos no disponibles; se usará búsqueda léxica:", type(error).__name__)
try:
    from langchain_core.documents import Document
    from langchain_core.embeddings import Embeddings
    from langchain_community.vectorstores import FAISS
    from langchain.agents import create_agent
    from langchain.tools import tool
    from langgraph.checkpoint.memory import InMemorySaver
    from langchain_openai import ChatOpenAI
    LANGCHAIN_DISPONIBLE = True
except Exception as error:
    Document = Embeddings = FAISS = create_agent = tool = InMemorySaver = ChatOpenAI = None
    LANGCHAIN_DISPONIBLE = False
    print("Framework de agente no disponible; se conserva el modo local:", type(error).__name__)
modelo_chat = None
if API_KEY and LANGCHAIN_DISPONIBLE:
    try:
        modelo_chat = ChatOpenAI(model=LLM_MODEL, base_url=LLM_BASE_URL,
                                 api_key=API_KEY, temperature=TEMPERATURA)
        print("Cliente remoto configurado; no se ha enviado ninguna solicitud.")
    except Exception as error:
        print("No se pudo preparar el cliente remoto:", type(error).__name__)

modules.json:   0%|          | 0.00/229 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/122 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/5.12k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/723 [00:00<?, ?B/s]

...

sentencepiece.bpe.model: downloading bytes:           |  0.00B            

tokenizer.json:   0%|          | 0.00/9.08M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/239 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Modelo de embeddings cargado en cuda


## 1. Objetivo y límites

S.A.F.E. organiza consultas de seguridad laboral y recupera pasajes con procedencia. El flujo incluye decisiones previas de urgencia, búsqueda semántica cuando el modelo local está disponible, respaldo léxico, planificación, herramientas y memoria de sesión. La API conversacional es opcional; sin clave puede revisarse el flujo local.

El corpus se compone de procedimientos internos simulados y resúmenes externos. Este prototipo no autoriza tareas, no reemplaza capacitación, protocolos ni supervisión.


In [ ]:
KNOWLEDGE_PATH = DIRECTORIO_SESION / "knowledge_base.json"
KNOWLEDGE_PATH.write_text(json.dumps([{"doc_id": "PTS-ALT-01", "titulo": "Procedimiento de Trabajo Seguro en Altura", "tipo": "procedimiento", "actividad": "trabajo en altura", "peligro": "caída de altura", "version": "3.2", "fecha": "2025-11-10", "contenido": "Para trabajos sobre 1.8 metros de altura se debe usar arnés de seguridad certificado con doble cola de vida y línea de anclaje independiente de la estructura de trabajo. Se debe inspeccionar el arnés y los conectores antes de cada uso, verificando ausencia de cortes, deshilachados o corrosión. El anclaje debe soportar al menos 2.270 kg por trabajador y estar ubicado por sobre el nivel de los hombros. Está prohibido trabajar en altura con viento superior a 50 km/h, lluvia intensa o visibilidad reducida. Toda plataforma de trabajo debe contar con barandas perimetrales y rodapiés.", "fuente_tipo": "interna"}, {"doc_id": "PTS-EXC-02", "titulo": "Procedimiento de Trabajo Seguro en Excavaciones", "tipo": "procedimiento", "actividad": "excavaciones", "peligro": "derrumbe, atrapamiento", "version": "2.0", "fecha": "2025-09-01", "contenido": "Toda excavación mayor a 1.5 metros de profundidad requiere entibación, talud estable o caja de protección, según el tipo de suelo determinado por el prevencionista. Antes de excavar se debe verificar la ausencia de redes de gas, agua, electricidad o telecomunicaciones mediante el procedimiento de detección de interferencias. El material extraído debe apilarse a una distancia mínima de 60 cm del borde de la excavación. Se prohíbe el ingreso de personal sin autorización del supervisor de excavación.", "fuente_tipo": "interna"}, {"doc_id": "PTS-IZA-03", "titulo": "Procedimiento de Trabajo Seguro en Izaje de Cargas", "tipo": "procedimiento", "actividad": "izaje de cargas", "peligro": "caída de carga, aplastamiento", "version": "2.1", "fecha": "2025-08-15", "contenido": "Toda maniobra de izaje debe ser dirigida por un rigger o aparejador calificado y comunicada mediante señales manuales o radio estandarizadas. Se debe verificar la carga máxima admisible de la grúa y de los elementos de izaje (eslingas, grilletes, ganchos) antes de cada maniobra. Está prohibido que personal transite o permanezca bajo cargas suspendidas. Se debe delimitar y señalizar el radio de giro de la grúa.", "fuente_tipo": "interna"}, {"doc_id": "PTS-ELEC-04", "titulo": "Procedimiento de Trabajo Seguro con Herramientas y Equipos Eléctricos", "tipo": "procedimiento", "actividad": "herramientas eléctricas", "peligro": "contacto eléctrico", "version": "1.4", "fecha": "2025-07-20", "contenido": "Las herramientas eléctricas portátiles deben contar con doble aislación o conexión a tierra efectiva, y ser alimentadas mediante un diferencial (protector de corriente residual) de 30 mA. No se deben usar cables con aislación dañada ni conexiones improvisadas. Antes de intervenir un circuito se debe aplicar el procedimiento de bloqueo y etiquetado (LOTO, ver PTS-LOTO-07). El trabajo eléctrico en presencia de humedad o lluvia está prohibido salvo con EPP dieléctrico certificado.", "fuente_tipo": "interna"}, {"doc_id": "PTS-MAQ-05", "titulo": "Procedimiento de Trabajo Seguro en Operación de Maquinaria Pesada", "tipo": "procedimiento", "actividad": "maquinaria", "peligro": "atropello, atrapamiento", "version": "2.3", "fecha": "2025-10-05", "contenido": "Solo puede operar maquinaria pesada (retroexcavadoras, cargadores frontales, grúas) personal con licencia interna vigente y curso de operación aprobado. Se debe realizar una revisión pre-uso documentada (checklist) antes de cada turno. Toda maniobra en reversa debe contar con apoyo de un señalero y alarma de retroceso audible. Está prohibido transportar personas en maquinaria no habilitada para ello.", "fuente_tipo": "interna"}, {"doc_id": "PTS-CONF-06", "titulo": "Procedimiento de Trabajo Seguro en Espacios Confinados", "tipo": "procedimiento", "actividad": "espacios confinados", "peligro": "asfixia, atmósfera tóxica", "version": "1.2", "fecha": "2025-06-12", "contenido": "Antes de ingresar a un espacio confinado (cámaras, pozos, estanques) se debe medir la atmósfera (oxígeno, gases inflamables y tóxicos) con equipo calibrado y obtener un permiso de trabajo firmado por el supervisor. Se requiere vigía permanente en el exterior con capacidad de rescate y comunicación continua con el interior. Está prohibido el ingreso si el oxígeno medido es inferior a 19.5% o superior a 23.5%.", "fuente_tipo": "interna"}, {"doc_id": "PTS-LOTO-07", "titulo": "Procedimiento de Bloqueo y Etiquetado (LOTO) de Energías Peligrosas", "tipo": "procedimiento", "actividad": "mantenimiento, herramientas eléctricas", "peligro": "energización accidental", "version": "1.0", "fecha": "2025-05-01", "contenido": "Antes de intervenir un equipo o circuito se deben identificar todas las fuentes de energía (eléctrica, hidráulica, neumática, mecánica), aislarlas físicamente y aplicar candado y tarjeta personal (LOTO). Cada trabajador que interviene debe colocar su propio candado; nadie puede retirar el candado de otra persona. Se debe verificar la ausencia de energía residual antes de comenzar la intervención.", "fuente_tipo": "interna"}, {"doc_id": "MAT-PEL-08", "titulo": "Matriz de Identificación de Peligros y Evaluación de Riesgos (IPER)", "tipo": "matriz", "actividad": "general", "peligro": "múltiples", "version": "4.0", "fecha": "2025-11-01", "contenido": "La matriz IPER de Triple T Corp identifica, por actividad y etapa de obra, los peligros asociados, su probabilidad, consecuencia y nivel de riesgo resultante, junto con las medidas de control jerarquizadas (eliminación, sustitución, controles de ingeniería, controles administrativos y EPP). Debe ser revisada cada vez que cambian las condiciones de la obra o se incorpora una nueva actividad, y está disponible para consulta de todo trabajador en la oficina de prevención.", "fuente_tipo": "interna"}, {"doc_id": "REG-OHS-09", "titulo": "Reglamento Interno de Orden, Higiene y Seguridad", "tipo": "reglamento", "actividad": "general", "peligro": "múltiples", "version": "5.0", "fecha": "2025-01-15", "contenido": "El Reglamento Interno establece las obligaciones y prohibiciones en materia de seguridad para todo trabajador de Triple T Corp, incluyendo el uso obligatorio de EPP, la prohibición de trabajar bajo efectos del alcohol o drogas, la obligación de reportar condiciones inseguras y el procedimiento disciplinario ante incumplimientos graves. Su cumplimiento es condición de permanencia en la obra.", "fuente_tipo": "interna"}, {"doc_id": "INST-EPP-10", "titulo": "Instructivo de Selección y Uso de Elementos de Protección Personal", "tipo": "instructivo", "actividad": "general", "peligro": "múltiples", "version": "3.0", "fecha": "2025-09-10", "contenido": "El EPP mínimo obligatorio en toda la obra es: casco de seguridad, zapatos de seguridad con puntera reforzada, chaleco reflectante y lentes de seguridad. Según la actividad se agregan elementos adicionales: arnés para trabajo en altura, protección auditiva en zonas de maquinaria, protección respiratoria en zonas con polvo o vapores, y guantes específicos según el riesgo (corte, químico, eléctrico). El EPP dañado debe reemplazarse de inmediato y no debe modificarse ni improvisarse.", "fuente_tipo": "interna"}, {"doc_id": "INST-CASCO-11", "titulo": "Instructivo Específico: Casco de Seguridad y Protección Craneana", "tipo": "instructivo", "actividad": "general", "peligro": "golpe en la cabeza", "version": "1.1", "fecha": "2025-04-01", "contenido": "El casco de seguridad debe cumplir norma de impacto y ajustarse firmemente con el barbiquejo en zonas de riesgo de caída de objetos o de altura. Debe reemplazarse tras cualquier impacto, aunque no presente daño visible, y como máximo cada 3 años desde su fabricación o 2 años desde su primer uso, según fecha impresa en el interior de la carcasa. No se deben perforar ni pintar con solventes.", "fuente_tipo": "interna"}, {"doc_id": "INST-ARNES-12", "titulo": "Instructivo Específico: Arnés de Seguridad y Sistemas Anticaídas", "tipo": "instructivo", "actividad": "trabajo en altura", "peligro": "caída de altura", "version": "1.3", "fecha": "2025-09-15", "contenido": "El arnés de cuerpo completo debe inspeccionarse antes de cada uso: costuras, hebillas, anillos D y cola de vida sin cortes ni desgaste. La cola de vida con absorbedor de impacto se conecta a un punto de anclaje certificado sobre el nivel de los hombros del trabajador. El arnés debe almacenarse limpio y seco, lejos de productos químicos y de la luz solar directa.", "fuente_tipo": "interna"}, {"doc_id": "INST-RESP-13", "titulo": "Instructivo Específico: Protección Respiratoria", "tipo": "instructivo", "actividad": "general", "peligro": "exposición a polvo, vapores", "version": "1.0", "fecha": "2025-03-10", "contenido": "En zonas con generación de polvo (corte, demolición) se debe usar mascarilla con filtro para material particulado; en zonas con vapores de solventes o pinturas se debe usar semi-máscara con cartucho para vapores orgánicos, seleccionado según la hoja de datos de seguridad del producto. El sello facial debe verificarse cada vez que se coloca el equipo.", "fuente_tipo": "interna"}, {"doc_id": "PROT-EMER-14", "titulo": "Protocolo de Emergencia, Evacuación e Incendio", "tipo": "protocolo", "actividad": "emergencia", "peligro": "incendio, emergencia general", "version": "2.2", "fecha": "2025-10-01", "contenido": "Al activarse la alarma de emergencia, todo trabajador debe suspender su actividad, dejar herramientas en posición segura y dirigirse por la ruta de evacuación señalizada hasta el punto de encuentro asignado, sin correr y sin devolverse por pertenencias. Los brigadistas designados coordinan el control inicial de incendios menores con extintores; ante fuego que no puede controlarse en los primeros segundos, se evacúa y se llama a Bomberos (132). El supervisor de turno realiza el conteo de personal en el punto de encuentro.", "fuente_tipo": "interna"}, {"doc_id": "PROT-DERR-15", "titulo": "Protocolo de Respuesta ante Derrame de Sustancias Químicas", "tipo": "protocolo", "actividad": "emergencia química", "peligro": "exposición química, incendio", "version": "1.1", "fecha": "2025-08-01", "contenido": "Ante un derrame, se debe alejar al personal no esencial del área, evitar fuentes de ignición si el producto es inflamable, y contener el derrame con el kit de contención disponible en el almacén de sustancias, solo si se cuenta con el EPP indicado en la hoja de datos de seguridad del producto. Si el derrame es de gran volumen, involucra un gas tóxico o hay personas expuestas, se debe activar de inmediato el protocolo de emergencia general (PROT-EMER-14) y notificar al supervisor y al prevencionista, sin intentar la contención por cuenta propia.", "fuente_tipo": "interna"}, {"doc_id": "PROC-ACC-16", "titulo": "Procedimiento de Notificación e Investigación de Accidentes del Trabajo", "tipo": "procedimiento", "actividad": "accidente", "peligro": "múltiples", "version": "2.0", "fecha": "2025-07-01", "contenido": "Todo accidente, incluso leve, debe reportarse de inmediato al supervisor directo, quien activa los primeros auxilios y, si corresponde, traslada al trabajador al organismo administrador de la Ley 16.744. Se debe preservar el lugar del accidente sin alterar evidencia hasta que el prevencionista realice la investigación. Todo accidente con tiempo perdido debe notificarse a la autoridad sanitaria y al organismo administrador dentro de las 24 horas.", "fuente_tipo": "interna"}, {"doc_id": "HDS-COMB-17", "titulo": "Hoja de Datos de Seguridad: Combustibles y Solventes de Obra", "tipo": "hoja_datos", "actividad": "sustancias químicas", "peligro": "incendio, exposición química", "version": "1.0", "fecha": "2025-05-20", "contenido": "Los combustibles y solventes almacenados en obra son líquidos inflamables de baja temperatura de inflamación. Deben almacenarse en bidones certificados, en un área ventilada, alejada de fuentes de calor y de zonas de tránsito, con extintor de polvo químico seco o CO2 cercano. En caso de contacto con la piel, lavar con agua abundante; en caso de inhalación, trasladar a la persona a un área ventilada y buscar atención médica si persisten los síntomas.", "fuente_tipo": "interna"}, {"doc_id": "PTS-CIRC-18", "titulo": "Procedimiento de Circulación Vehicular y Peatonal en Obra", "tipo": "procedimiento", "actividad": "circulación", "peligro": "atropello, colisión", "version": "1.5", "fecha": "2025-06-01", "contenido": "La velocidad máxima dentro de la obra es de 20 km/h para vehículos livianos y 10 km/h en zonas de tránsito peatonal. Los circuitos de circulación de vehículos y de peatones deben estar físicamente separados y señalizados. Todo vehículo debe usar alarma de retroceso y luces encendidas, y el conductor debe contar con licencia interna vigente.", "fuente_tipo": "interna"}, {"doc_id": "PTS-AND-19", "titulo": "Procedimiento de Trabajo Seguro con Andamios y Escalas", "tipo": "procedimiento", "actividad": "trabajo en altura", "peligro": "caída de altura, volcamiento", "version": "1.2", "fecha": "2025-09-05", "contenido": "Todo andamio debe ser armado por personal certificado, apoyado sobre superficie firme y nivelada, arriostrado y con barandas en plataformas sobre 1.8 metros. Debe inspeccionarse diariamente y contar con una tarjeta de habilitación visible (verde: apto, roja: fuera de servicio). Las escaleras portátiles deben asegurarse en su parte superior e inferior y sobrepasar en 1 metro el punto de apoyo superior.", "fuente_tipo": "interna"}, {"doc_id": "CAN-REP-20", "titulo": "Procedimiento y Canal de Reporte de Condiciones y Actos Inseguros", "tipo": "procedimiento", "actividad": "reporte", "peligro": "múltiples", "version": "1.0", "fecha": "2025-02-01", "contenido": "Cualquier trabajador puede reportar una condición o acto inseguro a través del canal interno habilitado (formulario en oficina de prevención, línea telefónica interna o buzón de obra), sin necesidad de identificarse. El prevencionista debe evaluar el reporte dentro de las 48 horas y comunicar las medidas adoptadas. El reporte oportuno no genera sanción disciplinaria para quien lo realiza de buena fe.", "fuente_tipo": "interna"}, {"doc_id": "PROT-PA-21", "titulo": "Protocolo de Primeros Auxilios Básicos en Obra", "tipo": "protocolo", "actividad": "primeros auxilios", "peligro": "múltiples", "version": "1.0", "fecha": "2025-04-15", "contenido": "Ante una persona lesionada, el primer paso es asegurar que el entorno sea seguro antes de acercarse, y avisar de inmediato al encargado de primeros auxilios y al supervisor. Ante una herida cortante con sangrado, se debe aplicar presión directa sobre la herida con un paño limpio o gasa del botiquín, elevar la extremidad afectada si es posible, y mantener la presión hasta que el sangrado se controle o llegue ayuda; si el sangrado es abundante o no se detiene, se debe trasladar a la persona a un centro de salud u organismo administrador de inmediato. No se debe mover a una persona con sospecha de lesión en cuello o columna salvo peligro inminente. Si la persona no responde y no respira con normalidad, se debe iniciar reanimación cardiopulmonar solo si se cuenta con formación certificada, y llamar de inmediato a los servicios de emergencia (SAMU 131). El botiquín de obra y el DEA (si existe) están ubicados en la oficina de prevención.", "fuente_tipo": "interna"}, {"doc_id": "REG-CAP-22", "titulo": "Registro y Programa de Capacitaciones en Seguridad Laboral", "tipo": "registro", "actividad": "capacitación", "peligro": "múltiples", "version": "1.0", "fecha": "2025-01-10", "contenido": "Todo trabajador nuevo recibe una inducción de seguridad antes de ingresar a obra, y capacitaciones periódicas según su actividad (altura, espacios confinados, izaje, manejo de químicos). El registro de capacitaciones y sus fechas de vencimiento es administrado por el área de prevención y es requisito para operar equipos o realizar tareas críticas.", "fuente_tipo": "interna"}, {"doc_id": "EXT-LEY16744", "titulo": "Ley N°16.744 — Seguro Social contra Riesgos de Accidentes del Trabajo y Enfermedades Profesionales", "institucion": "Superintendencia de Seguridad Social (SUSESO) / Biblioteca del Congreso Nacional", "url": "https://www.bcn.cl/leychile", "fecha_consulta": "2026-09-01", "version": "texto vigente", "actividad": "general", "peligro": "múltiples", "contenido": "Establece el seguro obligatorio contra accidentes del trabajo y enfermedades profesionales en Chile, las obligaciones del empleador en materia de prevención, y el deber de afiliación a un organismo administrador (mutualidad o ISL) que debe prestar atención médica y evaluar el accidente.", "tipo": "referencia externa", "fuente_tipo": "externa"}, {"doc_id": "EXT-DS594", "titulo": "Decreto Supremo N°594 — Reglamento sobre Condiciones Sanitarias y Ambientales Básicas en los Lugares de Trabajo", "institucion": "Ministerio de Salud de Chile", "url": "https://www.bcn.cl/leychile", "fecha_consulta": "2026-09-01", "version": "texto vigente", "actividad": "general", "peligro": "múltiples", "contenido": "Fija condiciones sanitarias y ambientales mínimas en los lugares de trabajo, incluyendo límites de exposición a agentes químicos y físicos, ventilación, y condiciones de orden y limpieza que todo empleador debe garantizar.", "tipo": "referencia externa", "fuente_tipo": "externa"}, {"doc_id": "EXT-CT-TITVII", "titulo": "Código del Trabajo — Título I, Libro II: De la Protección a los Trabajadores", "institucion": "Dirección del Trabajo", "url": "https://www.dt.gob.cl", "fecha_consulta": "2026-09-01", "version": "texto vigente", "actividad": "general", "peligro": "múltiples", "contenido": "Regula las obligaciones generales del empleador en materia de higiene y seguridad, el deber de protección eficaz de la vida y salud de los trabajadores, y el marco para los reglamentos internos de orden, higiene y seguridad.", "tipo": "referencia externa", "fuente_tipo": "externa"}, {"doc_id": "EXT-ISL-ALTURA", "titulo": "Guía Técnica de Trabajo en Altura", "institucion": "Instituto de Seguridad Laboral (ISL)", "url": "https://www.isl.gob.cl", "fecha_consulta": "2026-09-01", "version": "edición vigente", "actividad": "trabajo en altura", "peligro": "caída de altura", "contenido": "Orienta sobre las medidas de prevención recomendadas para trabajo en altura, incluyendo jerarquía de controles, sistemas de detención de caídas y criterios generales de capacitación, como complemento normativo a los procedimientos internos de cada empresa.", "tipo": "referencia externa", "fuente_tipo": "externa"}, {"doc_id": "EXT-NCH997", "titulo": "NCh997 — Elementos de Protección Personal: Clasificación, Requisitos y Marcado", "institucion": "Instituto Nacional de Normalización (INN)", "url": "https://www.inn.cl", "fecha_consulta": "2026-09-01", "version": "edición vigente", "actividad": "EPP", "peligro": "múltiples", "contenido": "Establece la clasificación de los elementos de protección personal, los requisitos mínimos de fabricación y el marcado que deben llevar para acreditar su certificación, usada como referencia al seleccionar EPP en los instructivos internos.", "tipo": "referencia externa", "fuente_tipo": "externa"}], ensure_ascii=False, indent=2), encoding="utf-8")
"""Demostrador local del asistente S.A.F.E.; usa solo la biblioteca estándar."""

import hashlib
import json
import math
import re
import unicodedata
from collections import deque
from datetime import datetime, timezone
from pathlib import Path
from typing import Any

ROOT = DIRECTORIO_SESION
UMBRAL_SCORE = 0.07
KNOWLEDGE_PATH = ROOT / "knowledge_base.json"
TRACE_PATH = ROOT / "trace.jsonl"
MEMORY_PATH = ROOT / "long_term_memory.json"

EMERGENCY_TERMS = ("no responde", "inconsciente", "electrocut", "incendio", "fuego", "explosión", "explosion", "atrapado", "atrapada", "hemorragia", "caída desde", "caida desde", "derrumbe", "colapso", "emergencia", "se está quemando", "se esta quemando")
STOP = {"el", "la", "los", "las", "un", "una", "de", "del", "y", "o", "en", "con", "por", "para", "que", "como", "qué", "cual", "cuál", "se", "debo", "puedo", "hay", "es", "al", "mi", "me", "nos", "sobre", "ante", "después", "antes"}


def normalize(text: str) -> str:
    text = unicodedata.normalize("NFKD", text.lower())
    return "".join(ch for ch in text if not unicodedata.combining(ch))


def terms(text: str) -> set[str]:
    return {w for w in re.findall(r"[a-z0-9]+", normalize(text)) if len(w) > 2 and w not in STOP}


def load_documents() -> list[dict[str, Any]]:
    with KNOWLEDGE_PATH.open(encoding="utf-8") as f:
        return json.load(f)


def plan(question: str) -> list[str]:
    """Plan simple y visible: clasifica, busca, contrasta evidencia y responde."""
    if is_emergency(question):
        return ["priorizar la seguridad inmediata", "derivar a los canales de emergencia del lugar", "entregar solo orientación general segura"]
    return ["identificar actividad y peligro", "buscar evidencia en la base local", "comprobar suficiencia y procedencia", "responder o reconocer falta de evidencia"]


def is_emergency(question: str) -> bool:
    q = normalize(question)
    return any(normalize(term) in q for term in EMERGENCY_TERMS)


def retrieve(question: str, limit: int = 4) -> list[dict[str, Any]]:
    q = terms(question)
    if not q:
        return []
    scored = []
    for doc in load_documents():
        doc_terms = terms(" ".join(str(doc.get(k, "")) for k in ("titulo", "actividad", "peligro", "contenido")))
        overlap = len(q & doc_terms)
        # Dice normalizado y leve prioridad interna; el número es una ordenación local, no una probabilidad.
        score = (2 * overlap / (len(q) + len(doc_terms))) if doc_terms else 0.0
        if doc.get("fuente_tipo") == "interna":
            score += 0.01
        if overlap:
            scored.append((score, doc))
    ordered = sorted(scored, key=lambda x: x[0], reverse=True)
    if not ordered:
        return []
    # Se descartan candidatos claramente más débiles para no mezclar temas distintos.
    best = ordered[0][0]
    relevant = [(score, doc) for score, doc in ordered if score >= max(0.065, best * 0.9)]
    return [dict(doc, score=round(score, 4)) for score, doc in relevant[:limit]]


def write_trace(question: str, answer: str, docs: list[dict[str, Any]], urgent: bool) -> None:
    """Registra metadatos mínimos; no persiste el texto literal de la pregunta."""
    event = {
        "ts_utc": datetime.now(timezone.utc).isoformat(timespec="seconds"),
        "consulta_hash": hashlib.sha256(question.encode("utf-8")).hexdigest()[:16],
        "urgente": urgent,
        "documentos": [d["doc_id"] for d in docs],
        "respuesta_hash": hashlib.sha256(answer.encode("utf-8")).hexdigest()[:16],
    }
    with TRACE_PATH.open("a", encoding="utf-8") as f:
        f.write(json.dumps(event, ensure_ascii=False) + "\n")


def save_memory(summary: str) -> None:
    """Memoria larga de preferencias/resúmenes explícitos, no de consultas completas."""
    items = []
    if MEMORY_PATH.exists():
        try:
            items = json.loads(MEMORY_PATH.read_text(encoding="utf-8"))
        except (json.JSONDecodeError, OSError):
            items = []
    items.append({"ts_utc": datetime.now(timezone.utc).isoformat(timespec="seconds"), "summary": summary[:240]})
    MEMORY_PATH.write_text(json.dumps(items[-100:], ensure_ascii=False, indent=2), encoding="utf-8")


class SafeAgent:
    """Ciclo local con memoria conversacional acotada y herramientas explícitas."""

    def __init__(self, memory_turns: int = 4):
        self.short_memory: deque[dict[str, str]] = deque(maxlen=memory_turns * 2)

    def invoke(self, question: str, log: bool = True) -> dict[str, Any]:
        urgent = is_emergency(question)
        follow_up = bool(self.short_memory and re.search(r"\b(y si|y cuando|y qué pasa|y que pasa|en ese caso|además)\b", normalize(question)))
        docs = retrieve(question)
        if follow_up:
            known = {d["doc_id"]: d for d in load_documents()}
            previous = [dict(known[doc_id], score=1.0) for doc_id in self.short_memory[-1].get("documentos", []) if doc_id in known]
            merged = {d["doc_id"]: d for d in previous + docs}
            docs = list(merged.values())[:4]
        if urgent:
            answer = ("La consulta puede describir una emergencia. Activa de inmediato el procedimiento de emergencia "
                      "del lugar y contacta a los servicios o responsables locales. No muevas a una persona lesionada "
                      "salvo que exista peligro inminente. Esta herramienta no sustituye la respuesta de emergencia.")
        elif not docs or docs[0]["score"] < UMBRAL_SCORE:
            answer = ("No encuentro evidencia suficiente en la base local para responder con seguridad. "
                      "Consulta el procedimiento vigente con supervisión o prevención antes de actuar.")
        else:
            top = docs[:2]
            source_list = "; ".join(f"{d['titulo']} ({d['doc_id']}, v{d.get('version', 's/d')})" for d in top)
            evidence = " ".join(d["contenido"] for d in top)
            answer = (f"Según la información disponible: {evidence}\n\n"
                      "Verifica que el procedimiento corresponda a la actividad y a su versión vigente. "
                      "Si las condiciones reales difieren o falta información, detén la tarea y consulta a supervisión o prevención.\n\n"
                      f"Fuentes: {source_list}.")
        self.short_memory.append({"consulta": question[:240], "respuesta": answer[:500], "documentos": [d["doc_id"] for d in docs]})
        result = {"respuesta": answer, "urgente": urgent, "documentos": docs, "plan": plan(question), "modo": "reglas_locales", "uso_memoria_corta": follow_up}
        if log:
            write_trace(question, answer, docs, urgent)
        return result


def tool_search(question: str) -> list[dict[str, Any]]:
    """Herramienta de consulta: devuelve pasajes con metadatos de origen."""
    return retrieve(question)


def tool_plan(question: str) -> list[str]:
    return plan(question)


def tool_record(question: str, result: dict[str, Any]) -> str:
    write_trace(question, result["respuesta"], result["documentos"], result["urgente"])
    return "Evento registrado con hash de consulta y sin texto literal."


### ¿Qué hace el agente?

En este prototipo, la búsqueda devuelve documentos con coincidencias léxicas normalizadas. El plan separa la intención, la búsqueda, la revisión de evidencia y la respuesta. En un entorno con LangChain instalado, la guía técnica muestra cómo envolver las mismas funciones como herramientas de un ciclo `create_agent`. La ruta principal se mantiene ejecutable sin dependencias externas.

In [ ]:
docs = load_documents()
print('Fuentes internas:', sum(d['fuente_tipo']=='interna' for d in docs))
print('Referencias externas simuladas:', sum(d['fuente_tipo']=='externa' for d in docs))
print('Campos de trazabilidad:', ', '.join(sorted(docs[0].keys())))

Fuentes internas: 22
Referencias externas simuladas: 5
Campos de trazabilidad: actividad, contenido, doc_id, fecha, fuente_tipo, peligro, tipo, titulo, version


### Recuperación semántica

Cuando los embeddings y LangChain están disponibles, se crea un índice vectorial FAISS a partir de los 27 textos con sus metadatos. Si el modelo no se descarga o el índice no se construye, el recuperador original mantiene la búsqueda léxica. En ambos casos, el score ordena candidatos y no es una probabilidad.

In [ ]:
retrieve_lexical = retrieve
UMBRAL_SCORE = 0.07
MIN_COINCIDENCIAS_TERMINOS = 2
MIN_SCORE_SEMANTICO = 0.75
vector_db = None
if modelo_embeddings is not None and LANGCHAIN_DISPONIBLE:
    try:
        class AdaptadorEmbeddings(Embeddings):
            def embed_documents(self, texts):
                return modelo_embeddings.encode(list(texts), normalize_embeddings=True).tolist()
            def embed_query(self, text):
                return modelo_embeddings.encode([text], normalize_embeddings=True)[0].tolist()
        documentos_lc = [Document(page_content=d["contenido"], metadata={k:v for k,v in d.items() if k!="contenido"}) for d in docs]
        vector_db = FAISS.from_documents(documentos_lc, AdaptadorEmbeddings())
        UMBRAL_SCORE = 0.55
        def retrieve(question, limit=4):
            resultados = vector_db.similarity_search_with_score(question, k=min(limit, len(docs)))
            terminos_consulta = terms(question)
            salida=[]
            for doc, distancia in resultados:
                score = max(0.0, 1.0-float(distancia)/2.0)
                terminos_documento = terms(" ".join(str(v) for v in doc.metadata.values()) + " " + doc.page_content)
                coincidencias = len(terminos_consulta & terminos_documento)
                if score >= UMBRAL_SCORE and (coincidencias >= MIN_COINCIDENCIAS_TERMINOS or score >= MIN_SCORE_SEMANTICO):
                    salida.append({**doc.metadata, "contenido":doc.page_content, "score":score})
            return salida
        print("FAISS index ready:", vector_db.index.ntotal, "fragments")
    except Exception as error:
        vector_db = None
        retrieve = retrieve_lexical
        print("Vector index unavailable; lexical search active:", type(error).__name__)
else:
    retrieve = retrieve_lexical
    print("Lexical search active for this session.")
print("Available documents:", len(docs))

FAISS index ready: 27 fragments
Available documents: 27


## 2. Planificación y herramientas

La planificación es deliberadamente interpretable: identifica actividad y peligro, busca evidencia, revisa si basta y responde o deriva. Las señales urgentes activan un flujo distinto. Las herramientas son `tool_search` (consulta), `tool_plan` (planificación) y `tool_record` (registro). La herramienta de registro guarda identificadores y hashes, no el texto original de la consulta.

In [ ]:
consulta_plan = '¿Qué revisar antes de usar un arnés en altura?'
print('Consulta:', consulta_plan)
for paso, accion in enumerate(plan(consulta_plan), 1):
    print(f'{paso}. {accion}')

Consulta: ¿Qué revisar antes de usar un arnés en altura?
1. identificar actividad y peligro
2. buscar evidencia en la base local
3. comprobar suficiencia y procedencia
4. responder o reconocer falta de evidencia


## 3. Memoria de corto y largo plazo

La memoria corta del agente guarda hasta cuatro turnos durante la sesión y se descarta al cerrar el proceso. La memoria larga es explícita y guarda únicamente un resumen que la persona decide persistir. En aplicaciones reales habría que definir consentimiento, retención, acceso y eliminación antes de almacenar cualquier dato identificable. Este cuaderno no guarda automáticamente las conversaciones completas.

In [ ]:
memory_agent = SafeAgent()
first = memory_agent.invoke('¿Qué equipo se requiere para trabajar sobre 1.8 metros de altura?', log=False)
follow_up = memory_agent.invoke('¿Y si llueve?', log=False)
print('Primera fuente:', [d['doc_id'] for d in first['documentos']])
print('Se usó contexto de la pregunta previa:', follow_up['uso_memoria_corta'])
print('Fuente conservada en el seguimiento:', [d['doc_id'] for d in follow_up['documentos']])

Primera fuente: ['PTS-ALT-01', 'PTS-AND-19']
Se usó contexto de la pregunta previa: True
Fuente conservada en el seguimiento: ['PTS-ALT-01', 'PTS-AND-19']


## 4. Prueba reproducible: respuesta con evidencia

La primera pregunta busca una instrucción del conjunto de ejemplo. Se inspeccionan los documentos recuperados y sus identificadores antes de interpretar la respuesta.

In [ ]:
agent = SafeAgent()
q1 = '¿Qué equipo se requiere para trabajar sobre 1.8 metros de altura?'
r1 = agent.invoke(q1)
print(r1['respuesta'])
print('\nDocumentos recuperados:')
for d in r1['documentos'][:3]: print('-', d['doc_id'], d['titulo'], 'score local=', d['score'])

Según la información disponible: Para trabajos sobre 1.8 metros de altura se debe usar arnés de seguridad certificado con doble cola de vida y línea de anclaje independiente de la estructura de trabajo. Se debe inspeccionar el arnés y los conectores antes de cada uso, verificando ausencia de cortes, deshilachados o corrosión. El anclaje debe soportar al menos 2.270 kg por trabajador y estar ubicado por sobre el nivel de los hombros. Está prohibido trabajar en altura con viento superior a 50 km/h, lluvia intensa o visibilidad reducida. Toda plataforma de trabajo debe contar con barandas perimetrales y rodapiés. Todo andamio debe ser armado por personal certificado, apoyado sobre superficie firme y nivelada, arriostrado y con barandas en plataformas sobre 1.8 metros. Debe inspeccionarse diariamente y contar con una tarjeta de habilitación visible (verde: apto, roja: fuera de servicio). Las escaleras portátiles deben asegurarse en su parte superior e inferior y sobrepasar en 1 metro el pu

## 5. Prueba de consulta que combina temas

La siguiente pregunta mezcla altura y condiciones ambientales. Se comprueba si aparecen fragmentos que aporten a la respuesta. La recuperación puede ser semántica con el modelo multilingüe o léxica de respaldo; una coincidencia no significa que el pasaje sea suficiente ni que el contenido esté vigente.

In [ ]:
q2 = '¿Qué medidas tomar para trabajar en altura cuando hay lluvia intensa y poco viento?'
r2 = agent.invoke(q2)
print(r2['respuesta'])
print('\nDocumentos recuperados:', [d['doc_id'] for d in r2['documentos']])

Según la información disponible: Para trabajos sobre 1.8 metros de altura se debe usar arnés de seguridad certificado con doble cola de vida y línea de anclaje independiente de la estructura de trabajo. Se debe inspeccionar el arnés y los conectores antes de cada uso, verificando ausencia de cortes, deshilachados o corrosión. El anclaje debe soportar al menos 2.270 kg por trabajador y estar ubicado por sobre el nivel de los hombros. Está prohibido trabajar en altura con viento superior a 50 km/h, lluvia intensa o visibilidad reducida. Toda plataforma de trabajo debe contar con barandas perimetrales y rodapiés.

Verifica que el procedimiento corresponda a la actividad y a su versión vigente. Si las condiciones reales difieren o falta información, detén la tarea y consulta a supervisión o prevención.

Fuentes: Procedimiento de Trabajo Seguro en Altura (PTS-ALT-01, v3.2).

Documentos recuperados: ['PTS-ALT-01']


## 6. Prueba fuera del alcance del corpus

Si no hay evidencia pertinente, el agente debe reconocerlo en vez de completar la respuesta con conocimiento general. La prueba se ejecuta contra el mismo corpus local.

In [ ]:
q3 = '¿Cuál es la capital de Chile?'
r3 = agent.invoke(q3)
print(r3['respuesta'])
print('Documentos recuperados:', [d['doc_id'] for d in r3['documentos']])

No encuentro evidencia suficiente en la base local para responder con seguridad. Consulta el procedimiento vigente con supervisión o prevención antes de actuar.
Documentos recuperados: []


## 7. Ruta urgente

Una consulta que menciona a una persona inconsciente requiere derivación prioritaria. La respuesta no intenta diagnosticar ni entregar una secuencia de primeros auxilios, porque esta versión no valida quién la ejecuta ni las condiciones del lugar.

In [ ]:
q4 = 'Una persona cayó de un andamio y no responde, ¿qué hago?'
r4 = agent.invoke(q4)
print(r4['respuesta'])
print('Clasificada como urgente:', r4['urgente'])
print('Plan:', r4['plan'])

La consulta puede describir una emergencia. Activa de inmediato el procedimiento de emergencia del lugar y contacta a los servicios o responsables locales. No muevas a una persona lesionada salvo que exista peligro inminente. Esta herramienta no sustituye la respuesta de emergencia.
Clasificada como urgente: True
Plan: ['priorizar la seguridad inmediata', 'derivar a los canales de emergencia del lugar', 'entregar solo orientación general segura']


## 8. Revisión de los resultados

Las salidas se producen al ejecutar las celdas, no se copian de resultados históricos. Conviene evaluar más casos, medir errores de recuperación y revisar falsos positivos de urgencia. La puntuación de recuperación sirve para ordenar candidatos y no es una probabilidad de acierto. Si en esta sesión no se cargaron embeddings, la ruta léxica no equivale a una búsqueda vectorial.

In [ ]:
tests = [
 ('con evidencia', r1, True),
 ('consulta combinada', r2, True),
 ('fuera de alcance', r3, False),
 ('posible emergencia', r4, True),
]
for nombre, resultado, esperado in tests:
    cumple = (resultado['urgente'] if nombre == 'posible emergencia' else bool(resultado['documentos']) if esperado else 'No encuentro evidencia suficiente' in resultado['respuesta'])
    print(f'{nombre}: resultado observado={cumple}; documentos={len(resultado["documentos"])}; urgente={resultado["urgente"]}')

con evidencia: resultado observado=True; documentos=2; urgente=False
consulta combinada: resultado observado=True; documentos=1; urgente=False
fuera de alcance: resultado observado=True; documentos=0; urgente=False
posible emergencia: resultado observado=True; documentos=0; urgente=True


## 9. Trazabilidad y memoria

Cada invocación genera una línea en `trace.jsonl` con fecha UTC, hashes y documentos. El archivo puede borrarse para reiniciar la demostración. Se evita guardar consultas crudas porque incluso una pregunta puede contener datos personales o sensibles. El ejemplo de memoria larga se deja desactivado para no persistir información sin necesidad.

In [ ]:
trace = BASE / 'trace.jsonl'
if trace.exists():
    rows = [json.loads(line) for line in trace.read_text(encoding='utf-8').splitlines() if line.strip()]
    print('Eventos registrados:', len(rows))
    if rows: print('Campos del último evento:', ', '.join(rows[-1].keys()))
else:
    print('Aún no existe registro; aparecerá después de invocar el agente.')

Eventos registrados: 4
Campos del último evento: ts_utc, consulta_hash, urgente, documentos, respuesta_hash


### Agente opcional con LangChain

El agente expone búsqueda, planificación y registro como herramientas. Su memoria de corto plazo se asocia a un identificador de sesión con `InMemorySaver` y se pierde al reiniciar el runtime. La ruta de urgencia local se mantiene antes de llamar al modelo.

In [ ]:
AGENTE_LANGCHAIN = None
if modelo_chat is not None and LANGCHAIN_DISPONIBLE:
    @tool
    def buscar_evidencia(consulta: str) -> str:
        """Recupera pasajes con su identificador, metadatos y texto."""
        return json.dumps([{k:d.get(k) for k in ("doc_id","titulo","version","contenido","score")} for d in retrieve(consulta)], ensure_ascii=False)
    @tool
    def planificar_consulta(consulta: str) -> str:
        """Devuelve los pasos priorizados para analizar una consulta."""
        return json.dumps(plan(consulta), ensure_ascii=False)
    @tool
    def registrar_evento(consulta: str, respuesta: str) -> str:
        """Registra hashes y documentos, sin escribir la consulta literal."""
        encontrados=retrieve(consulta)
        write_trace(consulta, respuesta, encontrados, is_emergency(consulta))
        return "Evento registrado con datos mínimos."
    try:
        AGENTE_LANGCHAIN = create_agent(
            model=modelo_chat,
            tools=[buscar_evidencia, planificar_consulta, registrar_evento],
            system_prompt=("Asistente de seguridad laboral. Usa solo la evidencia devuelta por buscar_evidencia, "
                           "cita doc_id, abstente cuando falte respaldo y deriva a supervisión o prevención. "
                           "No autorices tareas. Las urgencias se derivan antes de llamarte. Usa las herramientas."),
            checkpointer=InMemorySaver(),
        )
        print("Agente con herramientas y memoria de sesión listo.")
    except Exception as error:
        AGENTE_LANGCHAIN=None
        print("Agente remoto no inicializado:", type(error).__name__)

def consultar_agente_remoto(consulta, session_id="demo"):
    """Ejecuta la extensión remota; exige clave y modelo configurados."""
    if is_emergency(consulta):
        return {"respuesta": "La consulta puede describir una emergencia. Activa el procedimiento local y contacta a supervisión y a los servicios de emergencia; no esperes al asistente."}
    if AGENTE_LANGCHAIN is None:
        return {"respuesta": "La extensión remota no está disponible; usa `SafeAgent` para ejecutar la ruta local."}
    return AGENTE_LANGCHAIN.invoke({"messages":[{"role":"user","content":consulta}]},
                                   config={"configurable":{"thread_id":session_id}})
print("Clave disponible:", bool(API_KEY), "| extensión remota activa:", AGENTE_LANGCHAIN is not None)

Clave disponible: False | extensión remota activa: False


## 10. Integración opcional con un modelo

La versión local anterior es reproducible sin API. Para incorporar planificación adaptativa con un LLM, instala las dependencias indicadas en `requirements.txt` y configura `GROQ_API_KEY` en el entorno. Nunca pegues la clave en una celda ni la guardes en este cuaderno. Usa una clave individual y sigue las condiciones institucionales. En la arquitectura con LangChain, las funciones locales se exponen como herramientas; el modelo no debe anular la ruta de emergencia ni inventar respaldo documental.

### Comprobaciones de configuración y recuperación ante faltantes

Esta celda también verifica la ruta local sin clave, el comportamiento con una lista documental vacía y el estado de memoria tras reiniciarla. El código prueba el dispositivo que existe en esta sesión; GPU solo puede validarse si el runtime asignó una GPU.

In [ ]:
def recuperar_desde(documentos, consulta):
    """Prueba aislada para un conjunto disponible, incluso si está vacío."""
    if not documentos:
        return []
    q = terms(consulta)
    salida=[]
    for d in documentos:
        palabras=terms(" ".join(str(d.get(k,"")) for k in ("titulo","actividad","peligro","contenido")))
        if q and palabras:
            salida.append({**d,"score":len(q & palabras)/max(1,len(q))})
    return sorted(salida,key=lambda d:d["score"],reverse=True)[:K_RETRIEVAL]

def cargar_corpus_opcional(ruta):
    """Devuelve un corpus vacío si un archivo de prueba no existe o no es JSON válido."""
    try:
        dato=json.loads(Path(ruta).read_text(encoding="utf-8"))
        return dato if isinstance(dato,list) else []
    except (OSError, json.JSONDecodeError):
        return []

sin_api = SafeAgent().invoke("¿Qué indica la base sobre el uso de arnés?", log=False)
memoria_prueba = SafeAgent()
memoria_prueba.invoke("¿Qué revisar antes de trabajar en altura?", log=False)
memoria_prueba.short_memory.clear()
resultados_entorno = {
    "sin_api": (not API_KEY) or isinstance(sin_api["respuesta"], str),
    "archivo_de_datos_faltante": cargar_corpus_opcional(DIRECTORIO_SESION / "archivo_ausente.json") == [],
    "reinicio_memoria": len(memoria_prueba.short_memory) == 0,
    "cpu_o_gpu_detectado": DISPOSITIVO in {"cpu", "cuda"},
}
for nombre, ok in resultados_entorno.items():
    print(f"{'OK' if ok else 'REVISAR'} — {nombre}")
if DISPOSITIVO == "cuda" and modelo_embeddings is not None:
    prueba_gpu = modelo_embeddings.encode(["verificación de GPU"], convert_to_numpy=True)
    print("GPU: cálculo de embeddings completado; forma", prueba_gpu.shape)
else:
    print("GPU: no disponible en este runtime; no se atribuye una prueba de ejecución GPU.")
print("CPU: ruta seleccionada o disponible como alternativa.")

OK — sin_api
OK — archivo_de_datos_faltante
OK — reinicio_memoria
OK — cpu_o_gpu_detectado
GPU: cálculo de embeddings completado; forma (1, 768)
CPU: ruta seleccionada o disponible como alternativa.


## 11. Limitaciones y siguiente iteración

La base es pequeña y de demostración; sus resúmenes y procedimientos no son documentos oficiales. La búsqueda se basa en palabras compartidas, no embeddings. La regla de urgencia es simple y requiere evaluación contextual para controlar falsos positivos y omisiones. La memoria de sesión no se sincroniza entre procesos. El registro minimiza datos, pero todavía necesita política de acceso y retención para un uso real.

Como siguientes pasos: validar fuentes con prevención, diseñar un conjunto etiquetado de preguntas, medir recuperación y derivación, agregar control de versiones, revisar seguridad y privacidad, y probar la integración con proveedor en un entorno autorizado.

## 12. Cierre

Este prototipo permite observar un ciclo de agente con consulta, planificación, memoria breve, derivación y registro mínimo. Los resultados de esta ejecución describen solo las preguntas probadas y el corpus de ejemplo. La evaluación de exactitud, utilidad en terreno y cumplimiento normativo requiere fuentes reales, revisión experta y pruebas controladas.